# tth — train and evaluate on Kaggle (2× T4)

| GPU | Pipeline |
|---|---|
| 0 | eval-free start → **C**: chess SFT → chess GRPO → shell SFT → evals C0, C |
| 1 | eval **A** (bare) → **D** (control): table SFT → table GRPO → shell SFT → **B**: shell SFT → evals D, B |

Settings (right panel): **Accelerator = GPU T4 ×2**, **Internet = On**, **Input = your `tth` dataset (latest version)**.
Run cells 1→4 once, then the monitor. Shell evals use `TTH_SANDBOX=disposable-container` (no bwrap on Kaggle);
the tth product never honours it (tests/test_invariants.py). Settings are compute-limited and identical across arms.

### 1 · Copy the project from the dataset

In [ ]:
import os, shutil, subprocess, sys
W = '/kaggle/working/tth'; shutil.rmtree(W, ignore_errors=True); os.makedirs(W); os.chdir(W)
zips, roots = [], []
for d, dirs, files in os.walk('/kaggle/input', followlinks=True):
    zips += [os.path.join(d, f) for f in files if f.endswith('.zip')]
    if os.path.isdir(os.path.join(d, 'src', 'tth')):
        roots.append(d)
print('found zips:', zips, '\nfound extracted roots:', roots)
if roots:
    shutil.copytree(roots[0], W, dirs_exist_ok=True)
elif zips:
    subprocess.run(['unzip', '-qo', zips[0], '-d', W], check=True)
else:
    raise SystemExit('tth dataset not attached: right panel → Add Input → Your Datasets → tth')
sys.path[:0] = [W, W + '/src']
print(sorted(os.listdir(W)))
assert 'use_reentrant' in open('train/sft/train_lora.py').read(), 'OLD dataset version mounted: update the input to the latest version'

### 2 · Install dependencies

In [ ]:
!pip -q install peft python-chess tree-sitter tree-sitter-bash pydantic pyyaml 2>&1 | tail -1
!pip uninstall -y -q torchao  # Kaggle's torchao 0.10 breaks recent peft; tth does not use it
!apt-get -qq install -y stockfish > /dev/null 2>&1; ln -sf /usr/games/stockfish /usr/local/bin/stockfish; which stockfish
!nvidia-smi --query-gpu=index,name,memory.used,memory.total --format=csv
!wc -l data/chess/teacher_verified.jsonl data/chess/scripted_verified.jsonl data/table/scripted_verified.jsonl data/shell/teacher_verified.jsonl

### 3 · Quick check: parser + one generation from the base model (≈1 min, then frees the GPU)

In [ ]:
import os, sys
os.chdir('/kaggle/working/tth'); sys.path[:0] = ['/kaggle/working/tth', '/kaggle/working/tth/src']
import gc, torch
from tth.runtime.toolcalls import parse_qwen
assert parse_qwen('<|im_start|>\n{"name": "x", "arguments": {"a": 1}}\n<|im_end|>').tool_calls[0].name == 'x'
from transformers import AutoModelForCausalLM, AutoTokenizer
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-Coder-1.5B-Instruct')
m = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-Coder-1.5B-Instruct', torch_dtype=torch.float16).cuda()
enc = tok(['<|im_start|>user\nSay hi<|im_end|>\n<|im_start|>assistant\n'], return_tensors='pt').to('cuda')
print(repr(tok.decode(m.generate(**enc, max_new_tokens=10, do_sample=False)[0][enc.input_ids.shape[1]:])))
del m, enc; gc.collect(); torch.cuda.empty_cache()   # leave the GPUs empty for the pipelines
print('ok - model downloaded and cached')

### 4 · Launch both GPU pipelines (background)

In [ ]:
import os, sys
os.chdir('/kaggle/working/tth'); sys.path[:0] = ['/kaggle/working/tth', '/kaggle/working/tth/src']
import subprocess, time
for _ in range(40):
    used = [int(x) for x in subprocess.run('nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits',
            shell=True, capture_output=True, text=True).stdout.split()]
    if max(used) < 1000: break
    print('waiting for free GPUs, used MiB:', used); time.sleep(3)
os.makedirs('runs', exist_ok=True)
jobs = [subprocess.Popen(['bash', f'scripts/kaggle_gpu{i}.sh'], stdout=open(f'runs/gpu{i}.log', 'w'),
                         stderr=subprocess.STDOUT, start_new_session=True) for i in (0, 1)]
print('launched', [j.pid for j in jobs], '- now run the monitor')

### 5 · Live progress (refreshes every 10 s; safe to re-run after a disconnect). Expect ~75–90 min.

In [ ]:
import os, sys
os.chdir('/kaggle/working/tth'); sys.path[:0] = ['/kaggle/working/tth', '/kaggle/working/tth/src']
from train.monitor import watch
watch()

### 6 · Results table

In [ ]:
import os, sys
os.chdir('/kaggle/working/tth'); sys.path[:0] = ['/kaggle/working/tth', '/kaggle/working/tth/src']
!PYTHONPATH=src:. python -m evals.report

### 7 · Package → Output panel → download `tth_results.zip`

In [ ]:
!cd /kaggle/working/tth && zip -qr /kaggle/working/tth_results.zip adapters evals/results runs -x '*/peft/*' && ls -lh /kaggle/working/tth_results.zip

### If something fails
`!tail -40 /kaggle/working/tth/runs/gpu0.log` (or gpu1) and send the output. To stop everything: `!pkill -9 -f kaggle_gpu; pkill -9 -f 'train\.(sft|grpo)'; pkill -9 -f evals.run`